# 07 - Context Building
## Shakespeare Evidence Desk

- **Goal:** turn the ordered, reranked passages into a single grounded, cited **context** the generator can read — within a fixed **token budget**, with duplicates removed and every passage traceable to its source lines.
- **Experiment axis:** raw concatenation vs. deduplication plus token-budget packing vs. additionally expanding each passage with its same-scene parent context (the retrieval method, reranker, and candidate pool are held fixed; only the *assembly* changes).
- **Loads from Step 06:** `.artifacts/reranking-handoff.json` (the selected reranker), replayed on top of `.artifacts/retrieval-handoff.json` + `.artifacts/index.json` to rebuild each question's ordered candidates, plus `.artifacts/chunks.json` for parent-context lookup.
- **Exports:** `.artifacts/context-handoff.json` — the context-assembly decision for stage 08 (grounded generation).


## Why This Stage

*In general — what context building is and why a Retrieval-Augmented Generation (RAG) system needs it.* Retrieval found candidates and reranking ordered them, but the generator cannot consume a ranked list — it consumes one bounded prompt. Context building is the step that assembles those passages into the **context window**: it removes redundancy, optionally widens each passage to its surrounding scene, attaches citations, and packs the result to fit a **token budget**. It is the last stage that touches *evidence* before a single word of the answer is written.

The core decision this stage owns is *how to spend a fixed token budget on the best available evidence*. Spend it on raw concatenation and you risk overflowing the window, repeating overlapping chunks, and shipping uncited text. Spend it on deduplication plus budget-aware packing and you keep the most relevant, non-redundant passages with citations attached. Spend it further on parent-context expansion and each passage reads in full — at the cost of fitting fewer of them. Because none of these is free, the choice is *measured*: we keep the simplest assembly that preserves the evidence reranking surfaced, never overflows the budget, and cites every passage.


## Conceptual Foundations

### A ranked list is not yet a context

Reranking (stage 06) hands us an *ordered list of passages*. The generator (stage 08) does not read a list — it reads one **prompt**, a single block of text with a fixed size limit. Context building is the bridge: it packs the best passages into that block, drops redundancy, attaches citations so claims can be traced, and stops at a **token budget**. Everything the generator will ever "know" about the corpus for this question is decided here. Miss a passage at this step and no amount of prompting recovers it.

### The token budget is the hard constraint

A **token budget** is the maximum number of tokens of evidence we allow in the prompt. It exists because context windows are finite and because every token costs latency and money, but the deeper reason is quality: stuffing more passages in is not free. Models attend unevenly across a long context — relevant text buried in the middle of a large block is often under-used, the well-known **lost-in-the-middle** effect. So the budget is not just a cost cap; it is a *quality* lever that forces us to choose the few passages that matter instead of dumping the whole pool.

### Deduplication: overlap is a feature of chunking, a bug in context

Our chunks were cut with a 64-token overlap (notebook 02), and retrieval can return several chunks from the *same* scene. Left alone, the context repeats the same lines, spending budget on text the model has already seen and crowding out genuinely new evidence. **Deduplication** removes near-duplicate passages — here by comparing their word sets with the **Jaccard** overlap — so each slot in the budget carries fresh information.

### Parent context: the chunk versus the scene around it

A single 512-token chunk can land mid-exchange, with the line that answers the question making full sense only alongside the lines before and after it. **Parent context** (also called neighbor or window expansion) replaces a matched chunk with a larger surrounding span — its same-scene neighbor chunks, or the whole scene — so the generator sees the passage *in situ*. The trade is tokens: a richer passage is a bigger passage, and the budget is fixed, so expanding one passage means fitting fewer of them.

### Citations are assembled here, not generated later

Grounded answers must cite real passages. We do not ask the model to invent citation strings — we attach a **citation** to every passage as we pack it (work, act and scene, speakers, and the `chunk_id` that resolves back to source lines) and number it in the context. Stage 08 then references those numbers. Building citations at assembly time is what makes "separate fact from interpretation, cite the passage" enforceable rather than hoped for.

### What context building is *not* (clean stage boundaries)

- **Not retrieval (05) or reranking (06):** it does not search the corpus or re-score relevance; it consumes a fixed, already-ordered pool.
- **Not query handling (09):** it takes the question as written.
- **Not generation (08):** it assembles and cites evidence; it writes no answer and makes no claim.

### Common misconceptions and failure modes

- **"More context is better."** Past the point where the answer is covered, extra passages dilute attention and spend budget — often *lowering* answer quality (lost-in-the-middle).
- **"Just concatenate the top-k."** That ignores the budget (it can overflow the window), repeats overlapping chunks, and offers no citations — the naive control we measure against.
- **"Always expand to parent context."** Richer passages cost tokens; on a tight budget, expansion can push a second piece of evidence out, hurting multi-passage (comparison) questions.
- **"Dedup is cosmetic."** On an overlapped corpus it directly buys budget back; its value depends on how duplicate the pool actually is, which we measure rather than assume.


## 1. Load the Previous Selection

- **Purpose:** start from stage 06's exported decision, not hidden kernel state. Context building only *assembles* what retrieval + reranking already produced, so the retrieval method, reranker, candidate pool, index, and chunk metadata are fixed inputs here.
- **What the code does:** locates the repository root; reads `reranking-handoff.json` (the selected reranker and pool), `retrieval-handoff.json` (the selected search method), and `index.json` (rows with vectors + provenance); reconstructs the stage-05 search arms (Best Matching 25 (BM25), dense, hybrid Reciprocal Rank Fusion (RRF)) so the pool can be *replayed*; loads `chunks.json` and maps each chunk to its same-work **siblings** (for parent-context lookup); and rebuilds the **answer key** from the labeled questions (reusing the citation parser from notebooks 02–06) so Section 4 can grade which evidence survived assembly.
- **Fails loudly, early:** if any of the three handoffs or the index is missing, run notebooks 04–06 first. If the index was built on offline-fallback vectors (`embedding_backend != 'hosted'`), the notebook **stops here** — assembling context from fake retrieval is meaningless.
- **Expected result:** the index type and metric, the fixed upstream configuration (retrieval method, selected reranker, pool size, served top-k), how many works have sibling maps for parent expansion, and how many labeled questions resolved to an indexed scene. No network call happens here.


In [ ]:
import json
import math
import re
from collections import defaultdict
from pathlib import Path

# Work from the repository root so the notebook and the package read the same .env and artifacts.
ROOT = Path.cwd().resolve()
if not (ROOT / '.env.example').is_file() and (ROOT.parent / '.env.example').is_file():
    ROOT = ROOT.parent
if not (ROOT / '.env.example').is_file():
    raise RuntimeError('Use the repository as the notebook working directory.')

ARTIFACTS = ROOT / '.artifacts'
index_path = ARTIFACTS / 'index.json'
retrieval_handoff_path = ARTIFACTS / 'retrieval-handoff.json'
reranking_handoff_path = ARTIFACTS / 'reranking-handoff.json'
chunks_path = ARTIFACTS / 'chunks.json'
questions_path = ARTIFACTS / 'eval-questions.json'
for required in (index_path, retrieval_handoff_path, reranking_handoff_path):
    if not required.is_file():
        raise RuntimeError(f'Missing .artifacts/{required.name}. Run notebooks 04-06 first.')

index_payload = json.loads(index_path.read_text(encoding='utf-8'))
retrieval_handoff = json.loads(retrieval_handoff_path.read_text(encoding='utf-8'))
reranking_handoff = json.loads(reranking_handoff_path.read_text(encoding='utf-8'))
chunks_payload = json.loads(chunks_path.read_text(encoding='utf-8')) if chunks_path.is_file() else {'chunks': []}
evaluation_questions = json.loads(questions_path.read_text(encoding='utf-8')) if questions_path.is_file() else {'development': [], 'heldout': []}

index_type = index_payload['index_type']          # 'ivf' or 'exact', the stage-04 selection
metric = index_payload['metric']                  # 'inner_product' (unit vectors, so dot == cosine)
stored_dims = index_payload['dims']
embedding_backend = index_payload.get('embedding_backend', 'unknown')
records = index_payload['records']                # chunk_id, work_id, work, scenes, speakers, text, embedding

# Stop now if upstream vectors are offline-fallback: assembling context from fake retrieval is meaningless.
if embedding_backend != 'hosted':
    raise RuntimeError(
        f"index.json was built on the '{embedding_backend}' backend, not hosted Azure. "
        "Re-run notebooks 03-06 against Azure before building context.")

# ---- fixed upstream decisions this stage consumes (never re-decided here) ----
SELECTED_METHOD = retrieval_handoff['selected_method']                 # bm25 | dense | hybrid-rrf
RRF_K = retrieval_handoff.get('rrf_k') or 60
bm25_params = retrieval_handoff.get('bm25_params', {'k1': 1.5, 'b': 0.75})
SELECTED_RERANKER = reranking_handoff['selected_reranker']             # none | semantic | llm-listwise
RERANK_POOL = reranking_handoff.get('rerank_pool', 20)
TOP_K = reranking_handoff.get('top_k', 5)
LLM_SNIPPET_CHARS = reranking_handoff.get('llm_reranker', {}).get('snippet_chars', 320)

vectors = [r['embedding'] for r in records]
N = len(vectors)
work_titles = {r['work_id']: r['work'] for r in records}


def dot(a, b):
    # Score between two unit vectors; equals cosine similarity. Higher = more relevant.
    return sum(x * y for x, y in zip(a, b))


def l2_normalize(vector):
    length = math.sqrt(sum(value * value for value in vector))
    return [value / length for value in vector] if length else vector


_WORD_PATTERN = re.compile(r"[a-z0-9']+")


def tokenize(text):
    return _WORD_PATTERN.findall(text.lower())


# ---- replay stage-04 dense search + stage-05 BM25/RRF so the selected method rebuilds each pool ----
def exact_search_rows(query_vector, candidate_rows=None):
    pool = range(N) if candidate_rows is None else candidate_rows
    return sorted(pool, key=lambda i: dot(query_vector, vectors[i]), reverse=True)


if index_type == 'ivf':
    ivf = index_payload['ivf']
    ivf_centroids = ivf['centroids']
    ivf_lists = {int(cluster): rows for cluster, rows in ivf['lists'].items()}
    nprobe = index_payload['params']['nprobe']

    def dense_search(query_vector, k, candidate_rows=None):
        nearest = sorted(range(len(ivf_centroids)),
                         key=lambda c: dot(query_vector, ivf_centroids[c]), reverse=True)[:nprobe]
        pool = [i for c in nearest for i in ivf_lists.get(c, ())]
        if candidate_rows is not None:
            allowed = set(candidate_rows)
            pool = [i for i in pool if i in allowed]
        return exact_search_rows(query_vector, pool)[:k]
else:
    def dense_search(query_vector, k, candidate_rows=None):
        return exact_search_rows(query_vector, candidate_rows)[:k]


class BM25:
    """The same transparent Okapi BM25 keyword index built in notebook 05, reused to replay its pool."""

    def __init__(self, documents, term_saturation=1.5, length_strength=0.75):
        self.k1 = term_saturation
        self.b = length_strength
        self.tokenized_docs = [tokenize(text) for text in documents]
        self.doc_lengths = [len(tokens) for tokens in self.tokenized_docs]
        self.avg_len = (sum(self.doc_lengths) / len(self.tokenized_docs)) if self.tokenized_docs else 0.0
        self.term_counts_per_doc = []
        document_frequency = defaultdict(int)
        for tokens in self.tokenized_docs:
            term_counts = defaultdict(int)
            for term in tokens:
                term_counts[term] += 1
            self.term_counts_per_doc.append(term_counts)
            for term in term_counts:
                document_frequency[term] += 1
        total_documents = len(self.tokenized_docs)
        self.idf = {
            term: math.log(1 + (total_documents - doc_count + 0.5) / (doc_count + 0.5))
            for term, doc_count in document_frequency.items()
        }

    def search(self, query, top_k, candidate_rows=None):
        query_terms = [term for term in tokenize(query) if term in self.idf]
        candidate_indices = range(len(self.tokenized_docs)) if candidate_rows is None else candidate_rows
        scored_rows = []
        for row_index in candidate_indices:
            term_counts = self.term_counts_per_doc[row_index]
            doc_length = self.doc_lengths[row_index]
            score = 0.0
            for term in query_terms:
                term_count = term_counts.get(term, 0)
                if not term_count:
                    continue
                length_normalizer = self.k1 * (1 - self.b + self.b * doc_length / (self.avg_len or 1))
                score += self.idf[term] * (term_count * (self.k1 + 1)) / (term_count + length_normalizer)
            if score > 0:
                scored_rows.append((row_index, score))
        scored_rows.sort(key=lambda row_and_score: row_and_score[1], reverse=True)
        return [row_index for row_index, _ in scored_rows[:top_k]]


bm25 = BM25([record['text'] for record in records],
            term_saturation=bm25_params.get('k1', 1.5), length_strength=bm25_params.get('b', 0.75))


def reciprocal_rank_fusion(rankings, damping, top_k):
    fused_scores = defaultdict(float)
    for ranking in rankings:
        for rank, row_index in enumerate(ranking):
            fused_scores[row_index] += 1.0 / (damping + rank + 1)
    ranked_rows = sorted(fused_scores.items(), key=lambda row_and_score: row_and_score[1], reverse=True)
    return [row_index for row_index, _ in ranked_rows[:top_k]]


def retrieve_pool(query_text, query_vector, pool_size):
    """Replay the stage-05 selected method to produce the candidate pool reranking then ordered."""
    if SELECTED_METHOD == 'bm25':
        return bm25.search(query_text, pool_size)
    if SELECTED_METHOD == 'dense':
        return dense_search(query_vector, pool_size)
    dense_ranking = dense_search(query_vector, pool_size)
    keyword_ranking = bm25.search(query_text, pool_size)
    return reciprocal_rank_fusion([dense_ranking, keyword_ranking], RRF_K, pool_size)


# ---- parent/neighbor lookup: map each chunk to its siblings in the same work for context expansion ----
chunks_by_id = {c['chunk_id']: c for c in chunks_payload.get('chunks', [])}


def chunk_index(chunk_id):
    # chunk_id is 'work_id:strategy:index'; the trailing integer is the chunk's position within its work.
    tail = chunk_id.rsplit(':', 1)[-1]
    return int(tail) if tail.isdigit() else None


siblings_by_work = defaultdict(dict)   # work_id -> {position_index: chunk_id}
for chunk_id, chunk in chunks_by_id.items():
    position = chunk_index(chunk_id)
    if position is not None:
        siblings_by_work[chunk.get('work_id')][position] = chunk_id


# ---- answer key: resolve each labeled question's citation to the index rows holding that scene ----
ROMAN_VALUES = {'I': 1, 'V': 5, 'X': 10, 'L': 50, 'C': 100, 'D': 500, 'M': 1000}


def roman_to_int(value):
    if value is None:
        return None
    text = str(value).strip().upper()
    if text.isdigit():
        return int(text)
    total, previous = 0, 0
    for char in reversed(text):
        if char not in ROMAN_VALUES:
            return None
        current = ROMAN_VALUES[char]
        total += -current if current < previous else current
        previous = max(previous, current)
    return total or None


lowercase_title = {work_id: title.lower() for work_id, title in work_titles.items()}


def cited_locations(question):
    # Read the label's citation -> list of (work, act, scene); uses the label only, never the question's words.
    label_text = (question.get('question', '') + ' ' + question.get('expected_evidence', '')).lower()
    works = [work_id for work_id, title in lowercase_title.items() if title in label_text]
    act_scene_pairs = re.findall(r'act\s+(\d+|[ivxlc]+)\s*,?\s*scene\s+(\d+|[ivxlc]+)',
                                 question.get('expected_evidence', '').lower())
    acts_scenes = [(roman_to_int(act), roman_to_int(scene)) for act, scene in act_scene_pairs]
    return [(work_id, act, scene) for work_id in works for act, scene in acts_scenes]


def row_in_scene(record, work_id, act, scene):
    return record['work_id'] == work_id and any(a == act and s == scene for a, s in record['scenes'])


answer_key = []
for question in evaluation_questions.get('development', []):
    if not question.get('answerable'):
        continue
    locations = [(w, a, s) for w, a, s in cited_locations(question) if a is not None and s is not None]
    relevant_rows = {i for i, record in enumerate(records)
                     for (w, a, s) in locations if row_in_scene(record, w, a, s)}
    if relevant_rows:
        answer_key.append((question, relevant_rows))

print(f'Loaded {N:,} indexed rows ({index_type}, metric={metric}, {stored_dims} dims) from the {embedding_backend} backend.')
print(f'Fixed upstream: retrieval={SELECTED_METHOD}, reranker={SELECTED_RERANKER}, pool={RERANK_POOL}, served top-k={TOP_K}.')
print(f'Chunk siblings mapped for {len(siblings_by_work)} work(s) (parent-context lookup).')
print(f'Answer key: {len(answer_key)} labeled dev question(s) resolved to an indexed scene.')


## 2. Explain the Alternatives

A context builder is **good** when it preserves the evidence reranking surfaced (coverage), keeps the prompt free of repetition and noise (precision, redundancy), attaches a resolvable citation to every passage, and **never exceeds the token budget**. We **fix everything upstream** (the retrieval method, the reranker, the candidate pool) and vary only the *assembly strategy*.

### What we actually compare vs. what we only explain

| Method | Role here | Why |
|---|---|---|
| **Raw concatenation of top-k** | ✅ **compared** (control) | the naive assembly every real builder must beat — no dedup, no budget, no citations beyond order |
| **Deduplication + token-budget packing** | ✅ **compared** | drop near-duplicate passages, greedily pack by rank until the budget — the workhorse builder |
| **Deduplication + parent-context expansion + budget** | ✅ **compared** | additionally widen each passage to its same-scene neighbors so it reads in full |
| **Maximal Marginal Relevance (MMR) selection** | ○ **explained, not run** | re-selects passages trading relevance against novelty; dedup is a simpler special case for our overlap problem |
| **Contextual compression / summarization** | ○ **explained, not run** | an LLM rewrites passages to fit more in; adds a model call, cost, and a faithfulness risk we avoid before generation |
| **Sentence-window / small-to-big retrieval** | ○ **explained, not run** | retrieve small, expand to a parent at read time — the same idea as our parent arm, pushed into the retriever |

So three arms are compared head-to-head; the rest are explained so the choice makes sense.

### How each arm actually works (a line or two)

- **Raw concatenation** — take the served top-k reranked passages and join them in order, each with a numbered header. Zero logic: it may overflow the budget and repeat overlapping chunks, but it is the honest yardstick for whether assembly logic earns its keep.
- **Dedup + budget packing** — walk the ranked candidates; skip any passage whose word set overlaps an already-kept one past the **Jaccard** threshold (a near-duplicate); otherwise add it while the running token count stays under the budget. Produces a non-redundant, budget-safe, cited context.
- **Dedup + parent expansion** — the same walk, but each kept passage is first widened to include its immediate same-scene neighbor chunks (overlap lines removed, capped in tokens). Richer passages, fewer of them per budget.

### Why this is the experiment we run

Context building's whole premise is spending a fixed budget well. The genuine, measurable questions are **does assembly logic keep the evidence while fitting the budget** (dedup+budget vs. raw), and **is fuller per-passage context worth fitting fewer passages** (parent vs. no-parent). The budget, the Jaccard threshold, and the neighbor window are bounded settings, not competing arms.


## 3. Run Controlled Experiments

- **Fixed upstream:** the index, the selected retrieval method, the selected reranker, and the candidate pool are held constant; each question's **ordered candidates are produced once** (retrieve the pool, apply the stage-06 reranker) and frozen, so every builder assembles the *same* ranked rows.
- **What the code does:** sets the budget and policy knobs (token budget, Jaccard dedup threshold, parent neighbor window), builds the context toolkit (`tiktoken` token counting, Jaccard dedup, same-scene parent expansion, citation assembly), and registers the three builders — **raw**, **dedup+budget**, **dedup+parent**.
- **Query embedding & LLM calls:** questions are embedded with the same keyless Azure deployment and on-disk cache as notebooks 03–06 (usually **0** calls — already cached). Context building itself makes **no** model call; if the selected reranker is `llm-listwise`, its ordering is replayed from the stage-06 disk cache.
- **Bounded by design:** the token budget caps the context; parent expansion is capped per passage so one widened passage cannot swallow the budget. Nothing here fabricates text — passages are copied verbatim from the index and chunks.
- **Expected result:** the token budget and tokenizer, the candidate depth per question, the three registered builders, the number of questions embedded (and any live embedding calls), and the average ordered candidate pool size.


In [ ]:
import hashlib
import time

import tiktoken
from dotenv import dotenv_values
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import AzureOpenAI

CONTEXT_TOKEN_BUDGET = 2000     # the generator's evidence budget this stage must pack within
CONTEXT_POOL = RERANK_POOL      # ordered candidate depth context building may draw from (then budget decides)
DEDUP_THRESHOLD = 0.6           # token-Jaccard at/above which two passages count as near-duplicates
PARENT_NEIGHBORS = 1            # chunks on each side to pull as parent context (same scene only)
PARENT_MAX_TOKENS = 700         # cap on one expanded passage so parents cannot eat the whole budget

encoder = tiktoken.get_encoding('cl100k_base')   # the same tokenizer the chunker used in notebook 02


def count_tokens(text):
    return len(encoder.encode(text))


# ---- Azure clients (keyless, Entra identity); same on-disk caches as notebooks 03-06 ----
config = {key: value for key, value in dotenv_values(ROOT / '.env').items() if value}
token_provider = get_bearer_token_provider(DefaultAzureCredential(), 'https://cognitiveservices.azure.com/.default')
client = AzureOpenAI(
    azure_endpoint=config['AZURE_OPENAI_ENDPOINT'],
    api_version=config['AZURE_OPENAI_API_VERSION'],
    azure_ad_token_provider=token_provider,
)
embedding_deployment = config['AZURE_OPENAI_EMBEDDING_DEPLOYMENT']
chat_deployment = config['AZURE_OPENAI_CHAT_DEPLOYMENT']

EMBED_CACHE = ARTIFACTS / 'embedding-cache'
RERANK_CACHE = ARTIFACTS / 'rerank-cache'
EMBED_CACHE.mkdir(parents=True, exist_ok=True)
RERANK_CACHE.mkdir(parents=True, exist_ok=True)
embed_api_calls = 0


def embed_query(text):
    """Embed one query, reusing the 03-06 on-disk cache; call Azure only on a cache miss."""
    global embed_api_calls
    cache_file = EMBED_CACHE / (hashlib.sha256(f'{embedding_deployment}|{text}'.encode('utf-8')).hexdigest() + '.json')
    if cache_file.is_file():
        full_vector = json.loads(cache_file.read_text())
    else:
        full_vector = client.embeddings.create(model=embedding_deployment, input=[text]).data[0].embedding
        cache_file.write_text(json.dumps(full_vector))
        embed_api_calls += 1
    return l2_normalize(full_vector[:stored_dims])


# ---- replay the stage-06 selected reranker so context building starts from the SAME ordered candidates ----
RERANK_SYSTEM = (
    'You are a retrieval reranker for a Shakespeare evidence desk. '
    'Given a question and a numbered list of candidate passages, order the candidates '
    'from most to least relevant to answering the question. Judge only the text shown; '
    'do not use outside knowledge and do not invent passages. '
    'Return a JSON object {"ranking": [numbers]} listing every candidate number exactly once, best first.'
)


def apply_selected_reranker(query_text, query_vector, candidate_rows):
    """Reorder the pool exactly as stage 06 decided. For the LLM arm, replay its disk-cached ordering."""
    if SELECTED_RERANKER == 'none':
        return list(candidate_rows)
    if SELECTED_RERANKER == 'semantic':
        return sorted(candidate_rows, key=lambda row: dot(query_vector, vectors[row]), reverse=True)
    pool = list(candidate_rows)
    cache_key = hashlib.sha256(
        f"{chat_deployment}|{LLM_SNIPPET_CHARS}|{query_text}|{','.join(map(str, pool))}".encode('utf-8')).hexdigest()
    cache_file = RERANK_CACHE / (cache_key + '.json')
    if cache_file.is_file():
        order = json.loads(cache_file.read_text())
    else:
        numbered = '\n'.join(
            f"[{position}] ({records[row]['work']}) " + ' '.join(records[row]['text'].split())[:LLM_SNIPPET_CHARS]
            for position, row in enumerate(pool))
        response = client.chat.completions.create(
            model=chat_deployment,
            response_format={'type': 'json_object'},
            messages=[{'role': 'system', 'content': RERANK_SYSTEM},
                      {'role': 'user', 'content': f'Question: {query_text}\n\nCandidates:\n{numbered}'}],
        )
        parsed = json.loads(re.search(r'\{.*\}', response.choices[0].message.content or '{}', re.DOTALL).group(0))
        order = [int(n) for n in parsed.get('ranking', []) if isinstance(n, (int, float))]
        cache_file.write_text(json.dumps(order))
    seen, reranked = set(), []
    for position in order:
        if 0 <= position < len(pool) and position not in seen:
            seen.add(position)
            reranked.append(pool[position])
    reranked.extend(pool[position] for position in range(len(pool)) if position not in seen)
    return reranked


# ---- the context-building toolkit: dedup, parent expansion, and citation assembly ----
def token_set(text):
    return set(tokenize(text))


def jaccard(a, b):
    union = len(a | b)
    return len(a & b) / union if union else 0.0


def format_scene(scenes):
    if not scenes:
        return 'Unknown scene'
    act, scene = scenes[0]
    return f'Act {act}, Scene {scene}'


def parent_text(row):
    """Expand one chunk to its same-scene neighbors, overlap lines de-duplicated and the result token-capped."""
    record = records[row]
    work_id = record['work_id']
    position = chunk_index(record['chunk_id'])
    if position is None or work_id not in siblings_by_work:
        return record['text']
    pieces = []
    for offset in range(-PARENT_NEIGHBORS, PARENT_NEIGHBORS + 1):
        sibling_id = siblings_by_work[work_id].get(position + offset)
        sibling = chunks_by_id.get(sibling_id) if sibling_id else None
        if sibling and any(scene in record['scenes'] for scene in sibling.get('scenes', [])):
            pieces.append((position + offset, sibling['text']))
    pieces.sort()
    seen_lines, merged_lines = set(), []
    for _, text in pieces:
        for line in text.split('\n'):
            stripped = line.strip()
            if stripped and stripped in seen_lines:   # drop the overlap lines the chunker repeated
                continue
            if stripped:
                seen_lines.add(stripped)
            merged_lines.append(line)
    merged = '\n'.join(merged_lines).strip() or record['text']
    token_ids = encoder.encode(merged)
    return encoder.decode(token_ids[:PARENT_MAX_TOKENS]) if len(token_ids) > PARENT_MAX_TOKENS else merged


def passage_block(position, record, text):
    header = f"[{position}] {record['work']} - {format_scene(record['scenes'])}"
    if record['speakers']:
        header += f" ({', '.join(record['speakers'][:4])})"
    return f"{header}\n{text.strip()}"


def assemble(rows, texts):
    """Number the passages, build the context string, and attach a resolvable citation to each."""
    blocks, citations = [], []
    for position, (row, text) in enumerate(zip(rows, texts), start=1):
        record = records[row]
        blocks.append(passage_block(position, record, text))
        citations.append({
            'marker': position,
            'chunk_id': record['chunk_id'],
            'work': record['work'],
            'work_id': record['work_id'],
            'scene': format_scene(record['scenes']),
            'speakers': record['speakers'],
        })
    return {'rows': rows, 'context': '\n\n'.join(blocks), 'citations': citations}


# ---- the three context builders (control -> dedup+budget -> dedup+parent+budget) ----
def build_raw(ordered_rows):
    """Control: concatenate the served top-k reranked passages verbatim. No dedup, no budget, no parents."""
    chosen = list(ordered_rows[:TOP_K])
    return assemble(chosen, [records[row]['text'] for row in chosen])


def _pack(ordered_rows, expand):
    """Greedy dedup + token-budget packing, optionally expanding each kept passage to its parent context."""
    kept, kept_signatures, texts, used_tokens = [], [], [], 0
    for row in ordered_rows:
        signature = token_set(records[row]['text'])
        if any(jaccard(signature, previous) >= DEDUP_THRESHOLD for previous in kept_signatures):
            continue
        text = parent_text(row) if expand else records[row]['text']
        block_tokens = count_tokens(passage_block(len(kept) + 1, records[row], text)) + 2   # +2 for the blank-line join
        if used_tokens + block_tokens > CONTEXT_TOKEN_BUDGET:
            continue   # skip this one; a smaller later passage may still fit the remaining budget
        kept.append(row)
        kept_signatures.append(signature)
        texts.append(text)
        used_tokens += block_tokens
    return assemble(kept, texts)


def build_dedup_budget(ordered_rows):
    """Drop near-duplicates, then greedily pack passages in ranked order until the token budget is full."""
    return _pack(ordered_rows, expand=False)


def build_dedup_parent(ordered_rows):
    """Same as dedup+budget, but widen each kept passage to its same-scene neighbors first."""
    return _pack(ordered_rows, expand=True)


BUILDERS = {'raw': build_raw, 'dedup-budget': build_dedup_budget, 'dedup-parent': build_dedup_parent}

# Embed the answer-key questions once (cached), replay retrieval + the selected reranker, freeze ordered candidates.
query_vectors = [embed_query(question['question']) for question, _ in answer_key]
ordered_candidates = [
    apply_selected_reranker(question['question'], query_vector,
                            retrieve_pool(question['question'], query_vector, CONTEXT_POOL))
    for (question, _), query_vector in zip(answer_key, query_vectors)
]

avg_pool = sum(len(pool) for pool in ordered_candidates) / max(len(ordered_candidates), 1)
print(f'Token budget: {CONTEXT_TOKEN_BUDGET} tokens ({encoder.name}); candidate depth per question: up to {CONTEXT_POOL}.')
print(f'Registered builders: {", ".join(BUILDERS)}.')
print(f'Embedded {len(query_vectors)} question(s); {embed_api_calls} live embedding call(s) (rest from cache).')
print(f'Average ordered candidate pool: {avg_pool:.1f} rows (retrieval={SELECTED_METHOD}, reranker={SELECTED_RERANKER}).')


## Metrics in This Stage — What Each Number Means

Section 4 compares raw concatenation, deduplication plus budget packing, and dedup plus parent expansion on the **same ordered candidates**. The selection rule keeps the simplest builder that preserves evidence, never overflows the budget, and cites every passage.

| Metric | What it measures | How it is computed | What it signifies |
|---|---|---|---|
| Coverage at budget (evidence coverage) | whether assembly keeps the evidence reranking surfaced | relevant rows that made the final context divided by the question's total relevant rows | The primary metric — a builder must not drop evidence to fit the budget; higher is better. |
| Context precision | how much of the packed context is actually relevant | relevant passages divided by total passages in the context | Noise guard — low precision means budget spent on irrelevant passages; higher is better. |
| Average passages | how many passages the context holds | mean count of passages per assembled context | Density indicator; read alongside coverage and tokens, not alone. |
| Average tokens | how much of the window the context uses | mean of the tokenized context length (tiktoken cl100k_base) | Must stay under the budget; context against cost. |
| Budget utilization | how fully the budget is used | average tokens divided by the token budget | Near 1.0 is efficient; above 1.0 means overflow; direction depends on coverage. |
| Overflow rate | how often the context exceeds the budget | share of questions whose context tokens exceed the budget | Hard constraint — any overflow disqualifies a builder; must be zero. |
| Redundancy | how much repeated content remains | mean pairwise word-set Jaccard among the kept passages | Wasted budget on duplication; lower is better. |
| Citation integrity | whether every passage can be cited | share of passages whose citation resolves to a chunk and scene | Hard constraint — grounding requires it; must be 1.0. |
| p95 latency (ms) | the assembly cost | 95th percentile of the per-question build time | All builders are local and cheap; lower is better. |



## 4. Measure and Inspect

- **The lenses:** evidence **coverage at budget** and **context precision** (quality), **average passages / tokens / budget utilization** and **overflow rate** (the budget constraint), **redundancy** (wasted budget), **citation integrity** (grounding), and **p95 latency** (cost).
- **Coverage at budget — the primary metric.** The fraction of a question's relevant rows that survive into the final context. Assembly must not throw away evidence reranking worked to surface; a builder that fits the budget by dropping the answer has failed, however tidy its prompt.
- **Context precision and redundancy:** precision is relevant passages over total passages — budget spent on irrelevant text is noise that can trigger the lost-in-the-middle effect. Redundancy is the mean pairwise Jaccard among kept passages — on an overlapped corpus it shows how much budget duplication would have wasted, and how much dedup bought back.
- **Overflow rate and citation integrity are hard constraints.** Any context that exceeds the budget would be truncated by the generator (silent evidence loss), and any passage without a resolvable citation breaks grounding. Both must be clean for a builder to be eligible at all.
- **A worked assembly:** for one question we print the assembled `dedup-budget` context — its numbered citations and token count — so the packing and grounding are visible by eye.
- **Honest limit:** the labeled dev set is small, so single numbers are coarse. This pre-screens assembly strategies; the end-to-end judges are generation (08) and continuous evaluation (10).


In [ ]:
def near_duplicate_fraction(ordered_rows):
    """Share of pool candidates that are near-duplicates of an earlier one — what dedup would remove."""
    kept_signatures, duplicates = [], 0
    for row in ordered_rows:
        signature = token_set(records[row]['text'])
        if any(jaccard(signature, previous) >= DEDUP_THRESHOLD for previous in kept_signatures):
            duplicates += 1
        else:
            kept_signatures.append(signature)
    return duplicates / max(len(ordered_rows), 1)


def grade_context(built, relevant_rows):
    """Score one assembled context: coverage, precision, token size, redundancy, and citation integrity."""
    rows = built['rows']
    relevant_hits = sum(1 for row in rows if row in relevant_rows)
    signatures = [token_set(records[row]['text']) for row in rows]
    pairs = [jaccard(signatures[i], signatures[j])
             for i in range(len(signatures)) for j in range(i + 1, len(signatures))]
    integrity = bool(built['citations']) and all(
        citation['chunk_id'] and citation['scene'] != 'Unknown scene' for citation in built['citations'])
    tokens = count_tokens(built['context'])
    return {
        'passages': len(rows),
        'coverage': relevant_hits / max(len(relevant_rows), 1),
        'precision': relevant_hits / max(len(rows), 1),
        'tokens': tokens,
        'overflow': tokens > CONTEXT_TOKEN_BUDGET,
        'redundancy': sum(pairs) / len(pairs) if pairs else 0.0,
        'integrity': integrity,
    }


def evaluate_builder(builder):
    """Run one builder over every answer-key question and summarize quality, budget use, and cost."""
    coverage_total = precision_total = token_total = redundancy_total = 0.0
    passages_total = overflow_count = 0
    integrity_all = True
    latencies_ms = []
    for (question, relevant_rows), ordered in zip(answer_key, ordered_candidates):
        started = time.perf_counter()
        built = builder(ordered)
        latencies_ms.append((time.perf_counter() - started) * 1000)
        scored = grade_context(built, relevant_rows)
        coverage_total += scored['coverage']
        precision_total += scored['precision']
        token_total += scored['tokens']
        redundancy_total += scored['redundancy']
        passages_total += scored['passages']
        overflow_count += int(scored['overflow'])
        integrity_all = integrity_all and scored['integrity']
    questions = max(len(answer_key), 1)
    sorted_latencies = sorted(latencies_ms)
    p95 = sorted_latencies[min(len(sorted_latencies) - 1, int(0.95 * len(sorted_latencies)))] if sorted_latencies else 0.0
    return {
        'coverage': coverage_total / questions,
        'precision': precision_total / questions,
        'avg_passages': passages_total / questions,
        'avg_tokens': token_total / questions,
        'budget_utilization': (token_total / questions) / CONTEXT_TOKEN_BUDGET,
        'overflow_rate': overflow_count / questions,
        'redundancy': redundancy_total / questions,
        'citation_integrity': 1.0 if integrity_all else 0.0,
        'p95_latency_ms': p95,
    }


context_metrics = {name: evaluate_builder(builder) for name, builder in BUILDERS.items()}
pool_dup_fraction = sum(near_duplicate_fraction(pool) for pool in ordered_candidates) / max(len(ordered_candidates), 1)

print(f'Measured over {len(answer_key)} resolved dev question(s); budget={CONTEXT_TOKEN_BUDGET} tokens.')
print(f'Near-duplicate candidates in the pools (dedup would drop): {pool_dup_fraction:.0%}\n')
header = ('builder', 'cover', 'prec', 'passg', 'tokens', 'util', 'overflw', 'redund', 'cite', 'p95 ms')
print('{:<14} {:>6} {:>6} {:>6} {:>7} {:>6} {:>7} {:>7} {:>5} {:>7}'.format(*header))
for name, m in context_metrics.items():
    print('{:<14} {:>6.0%} {:>6.0%} {:>6.1f} {:>7.0f} {:>6.0%} {:>7.0%} {:>7.3f} {:>5.0%} {:>7.2f}'.format(
        name, m['coverage'], m['precision'], m['avg_passages'], m['avg_tokens'],
        m['budget_utilization'], m['overflow_rate'], m['redundancy'], m['citation_integrity'], m['p95_latency_ms']))

# Show one assembled context so the citations and budget packing are visible by eye.
if answer_key:
    sample = build_dedup_budget(ordered_candidates[0])
    print(f'\nSample context for "{answer_key[0][0]["question"][:60]}..." '
          f'({count_tokens(sample["context"])} tokens, {len(sample["citations"])} passages):')
    preview = sample['context'].split('\n')
    print('\n'.join(preview[:8]) + ('\n  ...' if len(preview) > 8 else ''))


## 5. Justify the Selection

- **Decision rule (data-driven, deterministic):** first keep only **feasible** builders — those that **never overflow** the budget, have **full citation integrity**, and do **not lose coverage** versus the raw control. Among the feasible, prefer **higher coverage**, then **higher context precision**, then **lower redundancy**, then the **simpler** builder. The rule is applied to the measured numbers, not asserted.
- **Why overflow and citations gate everything:** a context that exceeds the budget is silently truncated by the generator — evidence vanishes without warning — and a passage that cannot be cited breaks the whole grounding promise. These are not quality dials to trade off; they are correctness constraints, so they filter before anything else is compared.
- **Why coverage leads and simplicity breaks ties:** the point of assembly is to hand the generator the evidence, so the builder that keeps the most relevant rows wins; when two builders tie on evidence and noise, the simpler one (fewer moving parts, fewer tokens) is the honest choice.
- **Honest, data-dependent outcome (no pre-ordained winner):** raw concatenation of five ~500-token passages tends to **overflow** a 2000-token budget — which alone disqualifies it. Dedup plus budget packing fits the window and, because dense retrieval already ranked the cited scene at the top, usually keeps full coverage. Parent expansion reads richer but spends tokens, so it fits fewer passages and can **drop coverage on multi-evidence (comparison) questions** — valuable for deep single-scene lookups, costly for the cross-play questions this project centers on. We report whichever the numbers choose.
- **Expected result:** the selected builder with the numbers that justify it (coverage, precision, tokens, utilization), and each rejected alternative annotated with its measured reason.


In [ ]:
SIMPLICITY_ORDER = {'raw': 0, 'dedup-budget': 1, 'dedup-parent': 2}   # tie-break; lower = simpler
control_coverage = context_metrics['raw']['coverage']


def is_feasible(name):
    m = context_metrics[name]
    # Hard constraints: never overflow the budget, cite every passage, and do not lose coverage vs the control.
    return m['overflow_rate'] == 0.0 and m['citation_integrity'] == 1.0 and m['coverage'] >= control_coverage - 1e-9


def selection_key(name):
    m = context_metrics[name]
    # Among feasible builders: more coverage, then more precision, then less redundancy, then simpler.
    return (-m['coverage'], -m['precision'], m['redundancy'], SIMPLICITY_ORDER.get(name, 99))


feasible = [name for name in context_metrics if is_feasible(name)]
if feasible:
    selected_builder = min(feasible, key=selection_key)
else:
    # Nothing cleared the constraints: fall back to the lowest-overflow, highest-coverage builder and flag it.
    selected_builder = min(context_metrics,
                           key=lambda n: (context_metrics[n]['overflow_rate'], -context_metrics[n]['coverage']))

selected = context_metrics[selected_builder]
print(f'Selected builder: {selected_builder} '
      f'(budget={CONTEXT_TOKEN_BUDGET}, dedup>={DEDUP_THRESHOLD}, parent={"on" if selected_builder == "dedup-parent" else "off"})')
print(f"  coverage / precision:   {selected['coverage']:.0%} / {selected['precision']:.0%}")
print(f"  avg passages / tokens:  {selected['avg_passages']:.1f} / {selected['avg_tokens']:.0f} "
      f"({selected['budget_utilization']:.0%} of budget)")
print(f"  overflow / redundancy:  {selected['overflow_rate']:.0%} / {selected['redundancy']:.3f}")
print('\nAlternatives:')
for name, m in context_metrics.items():
    if name == selected_builder:
        continue
    if m['overflow_rate'] > 0:
        reason = f"overflows the budget on {m['overflow_rate']:.0%} of questions - disqualified"
    elif m['citation_integrity'] < 1.0:
        reason = 'a passage could not be cited - disqualified'
    elif m['coverage'] < control_coverage - 1e-9:
        reason = f"coverage {m['coverage']:.0%} drops below control {control_coverage:.0%}"
    else:
        reason = 'feasible but lower coverage / precision or more tokens than the selection'
    print(f"  {name}: cover={m['coverage']:.0%} prec={m['precision']:.0%} "
          f"tokens={m['avg_tokens']:.0f} util={m['budget_utilization']:.0%} - {reason}")


## 6. Export the Handoff

- **What gets exported:**
  - `.artifacts/context-handoff.json` — the decision record notebook 08 loads: the selected builder (`raw` / `dedup-budget` / `dedup-parent`), the token budget and tokenizer, the candidate depth, the dedup threshold, the parent-context policy, the packing rule, the citation format, the full builder comparison (coverage, precision, tokens, utilization, overflow, redundancy, citation integrity, p95 latency), the measured pool near-duplicate fraction, and a back-reference to the stage-06 reranking decision.
- **Why a config handoff, not a data dump:** the index and the upstream handoffs already hold the rows, vectors, and ordering; context building adds only a *policy* — how to turn ordered candidates into a cited, budgeted context. The handoff is that reproducible configuration plus the evidence for it. Notebook 08 replays retrieval + reranker + the selected builder to reproduce the exact context.
- **Provenance preserved:** every citation carries `chunk_id`, `work_id`, scene, and speakers, so generation (08) can quote and attribute back to the source lines.
- **Hygiene:** `.artifacts/` is gitignored and notebook outputs are stripped on commit — do not commit the handoff.
- **Expected result:** the handoff written, with the selected builder, budget, dedup threshold, and parent-context state printed.


In [ ]:
context_handoff_path = ARTIFACTS / 'context-handoff.json'

# context-handoff.json: the policy notebook 08 replays - how to turn ordered candidates into a cited,
# budgeted context - plus the measured comparison that justified it. The index + upstream handoffs hold the rest.
context_handoff = {
    'stage': '07-context-building',
    'selected_builder': selected_builder,                 # raw | dedup-budget | dedup-parent
    'token_budget': CONTEXT_TOKEN_BUDGET,
    'tokenizer': f'tiktoken/{encoder.name}',
    'candidate_pool': CONTEXT_POOL,
    'dedup': {'method': 'token-jaccard', 'threshold': DEDUP_THRESHOLD},
    'parent_context': {
        'enabled': selected_builder == 'dedup-parent',
        'strategy': 'same-scene-neighbor-window',
        'neighbors_each_side': PARENT_NEIGHBORS,
        'max_tokens': PARENT_MAX_TOKENS,
    },
    'packing': 'greedy-by-rank-within-budget',
    'citation_format': '[n] Work - Act X, Scene Y (speakers); citation carries chunk_id, work_id, scene, speakers',
    'selection_rule': 'feasible (no overflow, full citation integrity, coverage >= control), then max coverage, max precision, min redundancy, then simplicity',
    'reranking_reference': {                               # the fixed upstream this stage assembles
        'stage': reranking_handoff.get('stage', '06-reranking'),
        'selected_reranker': SELECTED_RERANKER,
        'rerank_pool': RERANK_POOL,
        'top_k': TOP_K,
        'retrieval_method': SELECTED_METHOD,
    },
    'candidates': context_metrics,                        # coverage, precision, tokens, overflow, redundancy, latency
    'pool_near_duplicate_fraction': pool_dup_fraction,
    'resolved_questions': len(answer_key),
    'production_target': {
        'module': 'production/retrieval/context_builder.py',
        'entrypoint': 'build_context(query, passages, token_budget)',
    },
}
context_handoff_path.write_text(json.dumps(context_handoff, ensure_ascii=False, indent=2), encoding='utf-8')

print(f'Wrote decision record to {context_handoff_path.relative_to(ROOT)}')
print(f'Selected: {selected_builder} | budget={CONTEXT_TOKEN_BUDGET} | dedup>={DEDUP_THRESHOLD} | '
      f'parent={"on" if selected_builder == "dedup-parent" else "off"}')
print('Reminder: .artifacts/ is gitignored and notebook outputs are stripped on commit - do not commit the handoff.')


## Observations and Open Questions

Honest notes from working through this stage — what the numbers do and do not prove, and what to revisit later.

- **Assembly can only keep what reranking surfaced.** Every coverage number here is bounded by the ordered pool. If a relevant passage never reached context building, no packing recovers it — the fix lives upstream in retrieval (05) or reranking (06), not here.

- **The budget is where the naive baseline breaks.** Five ~500-token passages do not fit a 2000-token window, so raw concatenation overflows and is disqualified on a correctness constraint, not a close quality call. That is the clearest argument for budget-aware packing: without it, the generator silently truncates evidence.

- **Dedup's value is proportional to pool redundancy.** On a corpus cut with overlap, the near-duplicate fraction is real and dedup buys budget back; on a pool that happens to be already distinct, dedup is a cheap safeguard that changes little. We report the measured fraction rather than assuming it is large.

- **Parent context trades breadth for depth.** Expanding passages reads better in isolation but fits fewer passages per budget, which hurts exactly the cross-play comparison questions this project centers on. It earns its place on single-scene depth questions; the handoff records the trade so stage 09 can revisit it per query type.

- **Coverage is the right lens for this stage.** Precision and redundancy matter, but the stage's job is to hand the generator the evidence; a tidy, non-redundant context that dropped the answer has failed. That is why the selection gates on coverage (plus the hard constraints), not on token efficiency alone.

- **How we will revisit.** Context building is a swappable policy: change the builder, budget, dedup threshold, or parent window, re-export the handoff, and re-run 08 on the **same** questions. Iterate on the **development** set; check the **held-out** set only periodically so evaluation stays honest.


## Completion Check and Next Notebook

- **Restart and run in order:** restart the kernel and run top to bottom to confirm this notebook depends only on the notebook 04/05/06 artifacts, not hidden state. Context building adds no retrieval and no generation; only question embedding touches Azure, and it is served from disk cache when available (if the selected reranker is `llm-listwise`, its ordering is replayed from cache too).
- **What we achieved:** three context builders (raw, dedup+budget, dedup+parent) run over the same ordered candidates and graded on coverage, precision, tokens, budget utilization, overflow, redundancy, citation integrity, and latency; a worked assembled context shown by eye; a deterministic, constraint-gated selection that keeps the simplest builder which preserves evidence, fits the budget, and cites every passage; and the exported context handoff.
- **Honest limits:** the labeled dev set is small, so coverage and precision are coarse; parent context's value is question-shape dependent (depth vs. breadth). The real judges are generation (notebook 08) and continuous evaluation (notebook 10).
- **Privacy and hygiene:** outputs are stripped on commit by `nbstripout`; `.artifacts/` is ignored. Do not commit the handoff.
- **Next stage — 08 Grounded Generation:** load `.artifacts/context-handoff.json`, replay retrieval + reranker + the selected builder to produce the cited context, then prompt the chat model to answer **strictly** from that context, cite the numbered passages, separate fact from interpretation, and abstain when the evidence cannot support an answer.


## Questions, Discussions, and Further Insights

The "Observations and Open Questions" cell records caveats about *our* numbers. These questions build the transferable intuition.

**Q1. Why not just concatenate the top passages and send them to the model?**
Because a bare concatenation ignores the three things this stage exists to handle. It can **overflow** the context window (the generator then truncates it silently, losing evidence), it **repeats** overlapping chunks (our chunker left 64-token overlaps), and it carries **no citations** the answer can reference. Dedup plus budget packing fixes all three for almost no cost, which is why the control is there to beat, not to ship.

**Q2. What is a token budget and why does it improve quality, not just cost?**
It is the cap on how many tokens of evidence enter the prompt. The obvious reason is cost and the finite window, but the subtler one is **lost-in-the-middle**: models attend unevenly across a long context, so relevant text buried in a wall of passages is under-used. A tighter budget forces selection of the few passages that matter, which often *raises* answer quality rather than lowering it.

**Q3. How does deduplication decide two passages are "the same"?**
It compares their word sets with the **Jaccard** overlap (intersection over union) and treats anything at or above a threshold (here 0.6) as a near-duplicate. This catches the overlap our chunker introduced and any re-chunked repeats, so each slot in the budget carries new information. It is deliberately simple and transparent; Maximal Marginal Relevance (MMR) generalizes the idea by trading relevance against novelty, but dedup is the special case our problem needs.

**Q4. What is parent context and when is it worth the tokens?**
A matched chunk can start mid-exchange; **parent context** replaces it with a wider same-scene span so the quoted line reads in situ. It is worth the tokens when a question needs *depth* in one place and the budget is loose. It is a poor trade when a question needs *breadth* — a cross-play comparison needs evidence from two plays, and a fatter single passage can crowd the second one out. That is exactly why we measure coverage rather than assume expansion helps.

**Q5. Why are overflow and citation integrity hard constraints instead of scored metrics?**
Because failing either is not "slightly worse," it is wrong. An overflowing context is truncated downstream, so evidence disappears unpredictably; an uncited passage cannot be attributed, so grounding collapses. Scored metrics trade off against each other; correctness constraints must simply hold, so they filter the candidate set before any trade-off is considered.

**Q6. Where do the citations come from, and why build them here?**
From the index metadata we carried since notebook 01 — `work`, `scene`, `speakers`, and the `chunk_id` that resolves to source lines. We attach them at assembly time and number each passage, so stage 08 references numbers it cannot get wrong rather than inventing citation strings. Building provenance into the context is what makes "cite the passage, separate fact from interpretation" enforceable.

**Q7. What did we produce, and what happens next?**
We exported `.artifacts/context-handoff.json` — the selected builder, the budget and tokenizer, the dedup and parent-context policy, the citation format, and the measured comparison that justified the choice. Notebook 08 (Grounded Generation) replays retrieval + reranker + this builder to produce the cited context, then prompts the chat model to answer **only** from it, cite passages, and abstain when the context cannot support an answer.
